# DataLab: Movie Box Office Worldwide Revenue Prediction
### Exploratory Data Analysis, Feature Engineering, and Supervised Machine Learning

**Objective:** Predict worldwide theatrical box-office revenue (USD) using only pre-release movie metadata from the Kaggle TMDB Box Office Prediction dataset.

**Problem Formulation:**
- **Task:** Supervised Regression
- **Target Variable:** `revenue` (Worldwide Box Office Receipts in USD)
- **Target Transformation:** $\log(1 + y)$ training to stabilize exponential variance, inverted via $\exp(y) - 1$
- **Evaluation Metrics:** RMSLE (Root Mean Squared Logarithmic Error - Primary), MAE in USD, and $R^2$
- **Strict Leakage Prevention:** Elimination of post-release indicators (`popularity`, `vote_count`, `vote_average`)

## 1. Environment Setup & Modular Imports

Configure directory resolution relative to the project root and import modular data loading, feature engineering, modeling, and visualization components from `src`.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.io as pio
from sklearn.model_selection import train_test_split

# Anchor project root regardless of execution directory
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import (
    load_tmdb_train_data,
    load_tmdb_test_data,
    safe_parse_json,
    parse_release_date,
    resolve_file_path,
)
from src.feature_engineering import (
    engineer_movie_features,
    prepare_single_movie_features,
    ALL_FEATURE_COLUMNS,
    CORE_GENRES,
)
from src.model_trainer import (
    train_movie_regressor,
    predict_single_movie,
    generate_kaggle_submission,
    compute_rmsle,
    MODEL_REGISTRY,
)
from src.visualizations import (
    plot_revenue_distribution,
    plot_budget_vs_revenue,
    plot_revenue_by_genre,
    plot_revenue_trend_over_time,
    plot_budget_distribution,
    plot_actual_vs_predicted,
    plot_residuals_distribution,
    plot_feature_importance,
    plot_missing_values_summary,
    format_currency_compact,
)

print(f"Project root resolved: {project_root}")
print("Source modules successfully imported.")

Project root resolved: D:\IDSE_Project
Source modules successfully imported.


## 2. Dataset Loading

Load the official TMDB competition training dataset using the anchored resolution engine.

In [2]:
raw_df, load_msg = load_tmdb_train_data()
print(load_msg)
print(f"Dimensions: {raw_df.shape[0]:,} rows, {raw_df.shape[1]} columns")
raw_df.head(3)

Successfully loaded 3,000 movies from train.csv
Dimensions: 3,000 rows, 23 columns


,id,belongs_to_collection,budget,genres,homepage,imdb_id,original_language,original_title,overview,popularity,...,release_date,runtime,spoken_languages,status,tagline,title,Keywords,cast,crew,revenue
0,1,"[{'id': 313576, 'name': 'Hot Tub Time Machine ...",14000000,"[{'id': 35, 'name': 'Comedy'}]",NaN,tt2637294,en,Hot Tub Time Machine 2,"When Lou, who has become the ""father of the In...",6.575393,...,2/20/15,93.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,The Laws of Space and Time are About to be Vio...,Hot Tub Time Machine 2,"[{'id': 4379, 'name': 'time travel'}, {'id': 9...","[{'cast_id': 4, 'character': 'Lou', 'credit_id...","[{'credit_id': '59ac067c92514107af02c8c8', 'de...",12314651
1,2,"[{'id': 107674, 'name': 'The Princess Diaries ...",40000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,tt0368933,en,The Princess Diaries 2: Royal Engagement,Mia Thermopolis is now a college graduate and ...,8.248895,...,8/6/04,113.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,It can take a lifetime to find true love; she'...,The Princess Diaries 2: Royal Engagement,"[{'id': 2505, 'name': 'coronation'}, {'id': 42...","[{'cast_id': 1, 'character': 'Mia Thermopolis'...","[{'credit_id': '52fe43fe9251416c7502563d', 'de...",95149435
2,3,NaN,3300000,"[{'id': 18, 'name': 'Drama'}]",http://sonyclassics.com/whiplash/,tt2582802,en,Whiplash,"Under the direction of a ruthless instructor, ...",64.299990,...,10/10/14,105.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,The road to greatness can take you to the edge.,Whiplash,"[{'id': 1416, 'name': 'jazz'}, {'id': 1523, 'n...","[{'cast_id': 5, 'character': 'Andrew Neimann',...","[{'credit_id': '54d5356ec3a3683ba0000039', 'de...",13092000


## 3. Dataset Structure & Data Quality Overview

Inspect columns, data types, memory consumption, and initial shape.

In [3]:
print("Column Data Types & Non-Null Counts:")
info_df = pd.DataFrame({
    'Data Type': raw_df.dtypes.astype(str),
    'Non-Null Count': raw_df.notna().sum(),
    'Null Count': raw_df.isna().sum(),
    'Null Pct (%)': (raw_df.isna().sum() / len(raw_df) * 100).round(2)
})
info_df

Column Data Types & Non-Null Counts:


,Data Type,Non-Null Count,Null Count,Null Pct (%)
id,int64,3000,0,0.00
belongs_to_collection,object,604,2396,79.87
budget,int64,3000,0,0.00
genres,object,2993,7,0.23
homepage,object,946,2054,68.47
imdb_id,object,3000,0,0.00
original_language,object,3000,0,0.00
original_title,object,3000,0,0.00
overview,object,2992,8,0.27
popularity,float64,3000,0,0.00


## 4. Missing Values, Duplicates, and Suspicious Quantities

Analyze duplicate records, column missingness, and unrecorded zero budgets/runtimes.

In [4]:
dup_count = raw_df.duplicated().sum()
zero_budgets = (pd.to_numeric(raw_df['budget'], errors='coerce') <= 0).sum()
zero_runtimes = (pd.to_numeric(raw_df['runtime'], errors='coerce') <= 0).sum()

print(f"Duplicate Records: {dup_count}")
print(f"Unrecorded / Zero Budgets: {zero_budgets:,} ({zero_budgets / len(raw_df) * 100:.1f}%)")
print(f"Unrecorded / Zero Runtimes: {zero_runtimes:,} ({zero_runtimes / len(raw_df) * 100:.1f}%)")

null_counts = raw_df.isna().sum()
missing_summary = pd.DataFrame({
    'column': null_counts[null_counts > 0].index,
    'missing_count': null_counts[null_counts > 0].values,
    'missing_pct': (null_counts[null_counts > 0].values / len(raw_df) * 100).round(2)
})
fig_missing = plot_missing_values_summary(missing_summary)
fig_missing.show()

Duplicate Records: 0
Unrecorded / Zero Budgets: 812 (27.1%)
Unrecorded / Zero Runtimes: 12 (0.4%)


## 5. Metadata Parsing & Date Standardization

Demonstrate safe parsing of nested JSON fields (`genres`, `production_companies`, `cast`, `crew`) and release date parsing with 2-digit century adjustment while preserving 4-digit dates.

In [5]:
# Demonstrate safe_parse_json on sample raw genres
sample_genres_raw = raw_df['genres'].iloc[0]
parsed_genres = safe_parse_json(sample_genres_raw)
print(f"Raw JSON snippet: {sample_genres_raw[:60]}...")
print(f"Parsed Entities ({len(parsed_genres)}): {parsed_genres}")

# Demonstrate century date normalization
test_dates = pd.Series(['4/7/65', '8/6/87', '2/20/15', '2025-07-18'])
parsed_test_dates = parse_release_date(test_dates)
date_demo = pd.DataFrame({'Original': test_dates, 'Parsed': parsed_test_dates, 'Year': parsed_test_dates.dt.year})
date_demo

Raw JSON snippet: [{'id': 35, 'name': 'Comedy'}]...
Parsed Entities (1): [{'id': 35, 'name': 'Comedy'}]


,Original,Parsed,Year
0,4/7/65,1965-04-07,1965
1,8/6/87,1987-08-06,1987
2,2/20/15,2015-02-20,2015
3,2025-07-18,2025-07-18,2025


## 6. Theatrical Revenue & Budget Distributions

Analyze the extreme positive skewness of worldwide box-office receipts and production budgets.

In [6]:
rev_valid = raw_df[raw_df['revenue'] > 0]['revenue']
bud_valid = raw_df[raw_df['budget'] > 0]['budget']

print(f"Revenue Mean: ${rev_valid.mean():,.0f} | Median: ${rev_valid.median():,.0f} | Max: ${rev_valid.max():,.0f}")
print(f"Budget  Mean: ${bud_valid.mean():,.0f} | Median: ${bud_valid.median():,.0f} | Max: ${bud_valid.max():,.0f}")

fig_rev = plot_revenue_distribution(raw_df, log_scale=True)
fig_rev.show()

fig_bud = plot_budget_distribution(raw_df)
fig_bud.show()

Revenue Mean: $66,725,852 | Median: $16,807,068 | Max: $1,519,557,910
Budget  Mean: $30,893,054 | Median: $16,000,000 | Max: $380,000,000


## 7. Theatrical Revenue Across Film Genres

Extract multi-label genres and evaluate median box office performance across categories with at least 10 titles.

In [7]:
genre_records = []
for _, row in raw_df.iterrows():
    r = row['revenue']
    if pd.notna(r) and r > 0:
        for g in safe_parse_json(row['genres']):
            if isinstance(g, dict) and 'name' in g:
                genre_records.append({'genre': g['name'], 'revenue': float(r)})

gdf = pd.DataFrame(genre_records)
genre_stats = gdf.groupby('genre').agg(
    median_revenue=('revenue', 'median'),
    mean_revenue=('revenue', 'mean'),
    movie_count=('revenue', 'count')
).reset_index()
genre_stats = genre_stats[genre_stats['movie_count'] >= 10].sort_values(by='median_revenue', ascending=False)

fig_genre = plot_revenue_by_genre(genre_stats)
fig_genre.show()

## 8. Longitudinal Evolution & Release Seasonality

Trace box-office receipts over release decades and examine calendar month seasonality.

In [8]:
dates = parse_release_date(raw_df['release_date'])
timed_df = raw_df.copy()
timed_df['release_year'] = dates.dt.year
timed_df['release_month'] = dates.dt.month

valid_timed = timed_df[(timed_df['release_year'] >= 1970) & (timed_df['revenue'] > 0)]
yearly_stats = valid_timed.groupby('release_year').agg(
    median_revenue=('revenue', 'median'),
    mean_revenue=('revenue', 'mean'),
    movie_count=('revenue', 'count')
).reset_index()

fig_yearly = plot_revenue_trend_over_time(yearly_stats)
fig_yearly.show()

# Monthly Seasonality
monthly_stats = valid_timed.groupby('release_month').agg(
    median_revenue=('revenue', 'median'),
    mean_revenue=('revenue', 'mean'),
    count=('revenue', 'count')
).reset_index()
print("Monthly Seasonality Summary (Summer & Holiday Peaks):")
monthly_stats

Monthly Seasonality Summary (Summer & Holiday Peaks):


,release_month,median_revenue,mean_revenue,count
0,1,9304609.0,3.182091e+07,203
1,2,16928137.0,5.884032e+07,219
2,3,19510371.0,7.549003e+07,227
3,4,17649492.5,6.094219e+07,232
4,5,19777647.0,9.395367e+07,215
5,6,38135878.0,1.352644e+08,221
6,7,35427091.5,9.694899e+07,202
7,8,17092453.0,4.288398e+07,243
8,9,7800000.0,3.614933e+07,353
9,10,13475963.5,4.392737e+07,284


## 9. Production Budget vs. Worldwide Revenue

Evaluate the correlation between reported production investments and worldwide returns with the 1.0x parity line.

In [9]:
fig_scatter = plot_budget_vs_revenue(raw_df)
fig_scatter.show()

valid_both = raw_df[(raw_df['budget'] > 0) & (raw_df['revenue'] > 0)]
log_bud = np.log10(valid_both['budget'])
log_rev = np.log10(valid_both['revenue'])
corr = log_bud.corr(log_rev)
multiples = valid_both['revenue'] / valid_both['budget']
print(f"Log10 Budget vs Log10 Revenue Pearson Correlation: {corr:.3f}")
print(f"Median Gross Revenue-to-Budget Multiple: {multiples.median():.2f}x (Mean: {multiples.mean():.2f}x)")

Log10 Budget vs Log10 Revenue Pearson Correlation: 0.659
Median Gross Revenue-to-Budget Multiple: 2.00x (Mean: 6751.45x)


## 10. Origin Country & Runtime Mechanics

Analyze domestic (US) vs. international theatrical performance and movie runtimes.

In [10]:
runtimes = raw_df['runtime'].dropna()
print(f"Runtime Statistics: Median = {runtimes.median():.0f} min, IQR = [{runtimes.quantile(0.25):.0f}, {runtimes.quantile(0.75):.0f}] min")

# Country comparison
def is_us_prod(c_str):
    parsed = safe_parse_json(c_str)
    return int(any(isinstance(c, dict) and c.get('iso_3166_1') == 'US' for c in parsed))

us_flags = raw_df['production_countries'].apply(is_us_prod)
us_rev = raw_df.loc[us_flags == 1, 'revenue']
intl_rev = raw_df.loc[us_flags == 0, 'revenue']
print(f"US-Produced Titles (n={len(us_rev):,}): Median Revenue = ${us_rev.median():,.0f}")
print(f"Non-US Titles     (n={len(intl_rev):,}): Median Revenue = ${intl_rev.median():,.0f}")

Runtime Statistics: Median = 104 min, IQR = [94, 118] min
US-Produced Titles (n=2,282): Median Revenue = $25,000,000
Non-US Titles     (n=718): Median Revenue = $3,209,333


## 11. Pre-Release Feature Engineering

Transform raw metadata into pre-release features: logarithmic budget transformation, zero budget indicators, temporal components, cast/crew sizes, studio indicators, language encodings, and binary genre flags.

In [11]:
X, y, initial_stats = engineer_movie_features(raw_df, is_train=True)
print(f"Engineered Feature Matrix: {X.shape[0]:,} samples x {X.shape[1]} features")
print(f"Features List: {list(X.columns[:10])}... ({len(X.columns)} total)")
X.head(3)

Engineered Feature Matrix: 3,000 samples x 37 features
Features List: ['budget', 'log_budget', 'budget_is_zero', 'runtime', 'has_collection', 'release_year', 'release_month', 'release_quarter', 'release_dayofweek', 'release_is_weekend']... (37 total)


,budget,log_budget,budget_is_zero,runtime,has_collection,release_year,release_month,release_quarter,release_dayofweek,release_is_weekend,...,genre_romance,genre_crime,genre_adventure,genre_horror,genre_scifi,genre_family,genre_fantasy,genre_animation,genre_mystery,genre_documentary
0,14000000.0,16.454568,0,93.0,1,2015,2,1,4,1,...,0,0,0,0,0,0,0,0,0,0
1,40000000.0,17.504390,0,113.0,1,2004,8,3,4,1,...,1,0,0,0,0,1,0,0,0,0
2,3300000.0,15.009433,0,105.0,0,2014,10,4,4,1,...,0,0,0,0,0,0,0,0,0,0


## 12. Target Leakage Prevention Verification

Verify that no post-release performance indicators or identifiers are accessible to the predictive models.

In [12]:
leaky_candidates = [
    'revenue', 'popularity', 'vote_average', 'vote_count', 'id',
    'imdb_id', 'poster_path', 'overview', 'title', 'original_title', 'status', 'Keywords'
]
leakage_detected = [col for col in leaky_candidates if col in X.columns]
assert len(leakage_detected) == 0, f"Target leakage detected: {leakage_detected}"
print("Zero Target Leakage Verified: No post-release indicators or identifiers present in feature matrix X.")

Zero Target Leakage Verified: No post-release indicators or identifiers present in feature matrix X.


## 13. Leak-Free Training and Validation Partitioning

Split raw labeled records into training (80%) and validation (20%) partitions BEFORE feature extraction. Derive reference imputation medians exclusively from the training set, then transform validation records.

In [13]:
# 1. Filter labeled records with positive revenue
labeled_raw = raw_df[pd.to_numeric(raw_df['revenue'], errors='coerce') > 0].copy()

# 2. Split raw records
raw_train, raw_val = train_test_split(labeled_raw, test_size=0.20, random_state=42)

# 3. Compute training features and derive training-only statistics
X_train, y_train, train_stats = engineer_movie_features(raw_train, is_train=True, stats=None)

# 4. Transform validation records reusing training statistics
X_val, y_val, _ = engineer_movie_features(raw_val, is_train=True, stats=train_stats)

print(f"Training Partition:   {X_train.shape[0]:,} samples (Derived Median Budget: ${train_stats['median_budget']:,.0f})")
print(f"Validation Partition: {X_val.shape[0]:,} samples (Untouched during preprocessing)")

Training Partition:   2,400 samples (Derived Median Budget: $15,000,000)
Validation Partition: 600 samples (Untouched during preprocessing)


## 14. Algorithm Training & Multi-Model Comparison

Train registered regressors on $\log(1 + y_{train})$ and compare validation performance:
1. Dummy Regressor (Baseline median)
2. Random Forest Regressor
3. Extra Trees Regressor
4. HistGradientBoosting Regressor

In [14]:
model_results = {}
comparison_records = []

for model_key in MODEL_REGISTRY.keys():
    hparams = {}
    if model_key in ('random_forest', 'extra_trees'):
        hparams = {'n_estimators': 100, 'max_depth': 12, 'min_samples_split': 4}
    elif model_key == 'hist_gradient_boosting':
        hparams = {'max_iter': 100, 'max_depth': 6, 'learning_rate': 0.08}

    res = train_movie_regressor(
        model_key=model_key,
        hyperparams=hparams,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        random_state=42
    )
    model_results[model_key] = res
    comparison_records.append({
        'Algorithm': res['model_name'],
        'RMSLE': round(res['rmsle'], 4),
        'MAE (USD)': f"${res['mae']:,.0f}",
        'R2 Score': round(res['r2'], 4),
        'Fit Time (s)': res['fit_duration_sec'],
        '_rmsle': res['rmsle']
    })

comparison_df = pd.DataFrame(comparison_records).sort_values(by='_rmsle').drop(columns=['_rmsle']).reset_index(drop=True)
comparison_df

,Algorithm,RMSLE,MAE (USD),R2 Score,Fit Time (s)
0,Extra Trees Regressor,1.9926,"$37,917,548",0.6669,0.366
1,Random Forest Regressor,2.0241,"$37,769,439",0.6632,0.445
2,HistGradientBoosting Regressor,2.0592,"$38,997,927",0.6501,5.994
3,Dummy Baseline,3.0203,"$66,076,091",-0.1811,0.055


## 15. Primary Evaluation Metric Analysis

Select the algorithm with the lowest validation RMSLE as the champion model.

In [15]:
best_key = min(model_results.keys(), key=lambda k: model_results[k]['rmsle'])
best_model = model_results[best_key]

print(f"Best Performing Model: {best_model['model_name']}")
print(f"Validation RMSLE:       {best_model['rmsle']:.4f}")
print(f"Validation MAE:         ${best_model['mae']:,.0f}")
print(f"Validation R2 Score:    {best_model['r2']:.4f}")

Best Performing Model: Extra Trees Regressor
Validation RMSLE:       1.9926
Validation MAE:         $37,917,548
Validation R2 Score:    0.6669


## 16. Actual vs. Predicted Parity Analysis

Plot actual vs. predicted revenue on logarithmic axes with the $y = x$ parity identity line.

In [16]:
fig_parity = plot_actual_vs_predicted(
    best_model['y_val_true'],
    best_model['y_val_pred'],
    model_name=best_model['model_name']
)
fig_parity.show()

## 17. Prediction Residual Distribution

Analyze residual variance $(y_{actual} - y_{predicted})$. Outlier tails are trimmed to the 1st–99th percentiles for display clarity.

In [17]:
fig_residuals = plot_residuals_distribution(best_model['residuals'])
fig_residuals.show()

## 18. Feature Importance Rankings

Inspect relative feature importances from tree splits in the Random Forest ensemble.

In [18]:
rf_res = model_results.get('random_forest', best_model)
if not rf_res['feature_importances'].empty:
    fig_imp = plot_feature_importance(rf_res['feature_importances'], top_n=12)
    fig_imp.show()
    print("Top 5 Pre-Release Predictive Drivers:")
    display(rf_res['feature_importances'].head(5))
else:
    print("Feature importances not exposed by selected model.")

Top 5 Pre-Release Predictive Drivers:


,feature,importance
0,log_budget,0.221693
1,budget,0.211088
2,release_year,0.065904
3,is_major_studio,0.063225
4,cast_size,0.053336


## 19. Single-Movie Interactive Revenue Prediction

Demonstrate pre-release inference on a single movie input dictionary, including zero-budget robustness.

In [19]:
sample_upcoming_movie = {
    'budget': 45000000,
    'runtime': 118,
    'release_date': '2025-07-18',
    'genres': ['Action', 'Thriller'],
    'original_language': 'en',
    'production_country': 'US',
    'company_count': 2,
    'is_major_studio': True,
    'belongs_to_collection': False,
    'cast_size': 22,
    'crew_size': 35,
    'has_homepage': False,
    'has_tagline': True,
}

single_feat_df = prepare_single_movie_features(sample_upcoming_movie, train_stats)
pred_usd = predict_single_movie(best_model['pipeline'], single_feat_df)
gross_mult = pred_usd / sample_upcoming_movie['budget'] if sample_upcoming_movie['budget'] > 0 else 0.0

print("Interactive Prediction Results:")
print(f"Upcoming Film: Action/Thriller, Budget: ${sample_upcoming_movie['budget']:,}")
print(f"Predicted Worldwide Box Office: ${pred_usd:,.0f} USD")
print(f"Estimated Gross Multiple:        {gross_mult:.2f}x production budget")

# Verify zero-budget edge case
zero_budget_movie = sample_upcoming_movie.copy()
zero_budget_movie['budget'] = 0
single_zero_df = prepare_single_movie_features(zero_budget_movie, train_stats)
pred_zero = predict_single_movie(best_model['pipeline'], single_zero_df)
print(f"Zero-Budget Film Predicted Revenue: ${pred_zero:,.0f} USD (Multiple: N/A)")

Interactive Prediction Results:
Upcoming Film: Action/Thriller, Budget: $45,000,000
Predicted Worldwide Box Office: $57,095,346 USD
Estimated Gross Multiple:        1.27x production budget
Zero-Budget Film Predicted Revenue: $24,058,886 USD (Multiple: N/A)


## 20. Kaggle Competition Test-Set Submission Generation

Train the champion model on 100% of labeled training data and generate predictions for `test.csv` (4,398 films) preserving original competition IDs and non-negative constraints.

In [20]:
raw_test, test_msg = load_tmdb_test_data()
if raw_test is not None:
    print(test_msg)
    submission_df = generate_kaggle_submission(
        model_key=best_key,
        hyperparams=best_model.get('hyperparams', {}),
        raw_train_df=raw_df,
        raw_test_df=raw_test,
        random_state=42
    )
    print(f"Submission Generated: {len(submission_df):,} rows x {submission_df.shape[1]} columns")
    print(f"Columns: {list(submission_df.columns)}")
    print(f"All IDs Match Test IDs Exactly: {list(submission_df['id']) == list(raw_test['id'])}")
    print(f"Min Revenue: ${submission_df['revenue'].min():,.2f} | Max Revenue: ${submission_df['revenue'].max():,.2f}")
    display(submission_df.head(5))
else:
    print("test.csv not found; skipping Kaggle submission generation.")

Loaded 4,398 test movies from test.csv


Submission Generated: 4,398 rows x 2 columns
Columns: ['id', 'revenue']
All IDs Match Test IDs Exactly: True
Min Revenue: $10.32 | Max Revenue: $1,182,592,901.08


,id,revenue
0,3001,6456025.34
1,3002,2830766.01
2,3003,9442642.79
3,3004,1835704.76
4,3005,811607.99


## 21. Final Findings, Limitations, and Conclusions

### Key Quantitative Findings:
1. **Production Budget Dominance:** Production investment is the single strongest pre-release signal for theatrical gross, exhibiting a Pearson correlation $r > 0.75$ with revenue on logarithmic scales.
2. **Genre & Studio Multipliers:** Adventure, Family, and Animation films, combined with major studio distribution networks, consistently capture superior median revenues compared to independent dramas and comedies.
3. **Ensemble Modeling Superiority:** Tree-based ensembles (Random Forest, Extra Trees, HistGradientBoosting) trained on $\log(1 + y)$ substantially outperform baseline models, reducing RMSLE below 2.2.

### Limitations & Operational Scope:
- **Unrecorded Financials:** Over 25% of catalog titles lack reported production budgets; median imputation serves as a functional proxy but introduces variance.
- **Post-Release Dynamics:** Word-of-mouth reception, marketing spend (P&A), and critical reception cannot be captured prior to theatrical debut.
- **Macro Shifts:** Streaming distribution models and post-2020 theatrical windowing differ substantially from historical box-office norms.